In [1]:
import pandas as pd
df = pd.read_csv('data.csv')
df.info()
df.isnull().sum()
df

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 901 entries, 0 to 900
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   id                901 non-null    object
 1   topic             901 non-null    object
 2   subtopic          901 non-null    object
 3   persona           901 non-null    object
 4   opening_question  901 non-null    object
 5   messages          901 non-null    object
dtypes: object(6)
memory usage: 42.4+ KB


,id,topic,subtopic,persona,opening_question,messages
0,451c070a-78de-4a13-bafa-d4bdd697af9e,Space Law,Space Law Basics,"a busy person, needing quick and efficient ans...",Basics of Space Law? Role of international tre...,"[{'role': 'user', 'content': 'Basics of Space ..."
1,2571369a-53ac-4dba-af23-6bf0c85ce3af,Space Law,1998 ISS agreement,a researcher with strong analytical mind and e...,Could you provide a detailed analysis of the 1...,"[{'role': 'user', 'content': 'Could you provid..."
2,d6455bcf-a808-4087-8b2c-2da820646bd5,Space Law,1998 ISS agreement,a professional in the domain seeking for techn...,What are the specific impacts of the 1998 ISS ...,"[{'role': 'user', 'content': 'What are the spe..."
3,5d1b1697-e276-4f6b-9e5a-1e724e483e4a,Space Law,1998 ISS agreement,"a professional science, technical, or engineer...",What are the main operational protocols define...,"[{'role': 'user', 'content': 'What are the mai..."
4,ae003606-8c44-4c8f-88b0-a5aca9967d3e,Space Law,1998 ISS agreement,a person whith very little technical understan...,Can you explain how the 1998 space station dea...,"[{'role': 'user', 'content': ""Can you explain ..."
...,...,...,...,...,...,...
896,fb8a09a9-0eed-4271-9f03-721ec631cc2b,Telecommunication,Deep Space Communications,"an expert of the domain, asking for very speci...",Could you elaborate on the fundamental princip...,"[{'role': 'user', 'content': 'Could you elabor..."
897,53f0f242-f27e-4dfe-a091-bc1290b0dbaa,Telecommunication,Deep Space Communications,"an expert of the domain, asking for very speci...",Could you elaborate on the role of varying ele...,"[{'role': 'user', 'content': 'Could you elabor..."
898,0e99050b-eb03-4199-a7c2-b961fd098883,Telecommunication,Deep Space Communications,"a basic technician, with basic language enquir...",What are the main problems with setting up and...,"[{'role': 'user', 'content': 'What are the mai..."
899,ae8c3763-1707-41fd-8ee1-bd18a0cf6708,Telecommunication,Deep Space Communications,a college student who uses slang and abbreviat...,How does the design of antennas affect how goo...,"[{'role': 'user', 'content': 'How does the des..."


In [2]:
import re, ast


def _norm(s: str) -> str:
    if not isinstance(s, str):
        s = "" if pd.isna(s) else str(s)
    s = s.replace("\u200b", " ")
    s = re.sub(r"[ \t]+", " ", s)
    s = re.sub(r"\n{3,}", "\n\n", s)
    return s.strip()


rows = []
for i, row in df.iterrows():
    try:
        msgs = ast.literal_eval(row["messages"])
    except Exception:
        continue

    current_q = None
    for m in msgs:
        role = m.get("role")
        content = _norm(m.get("content", ""))
        if not content:
            continue
        if role == "user":
            current_q = content
        elif role == "assistant" and current_q:
            rows.append({
                "topic": row["topic"],
                "subtopic": row["subtopic"],
                "persona": row["persona"],
                "question": current_q,
                "answer": content,
            })
            current_q = None

qa_df = pd.DataFrame(rows)
qa_df


,topic,subtopic,persona,question,answer
0,Space Law,Space Law Basics,"a busy person, needing quick and efficient ans...",Basics of Space Law? Role of international tre...,### Basics of Space Law\n\nSpace Law encompass...
1,Space Law,Space Law Basics,"a busy person, needing quick and efficient ans...",Can you provide examples of specific national ...,"Certainly, here are examples of specific natio..."
2,Space Law,1998 ISS agreement,a researcher with strong analytical mind and e...,Could you provide a detailed analysis of the 1...,The 1998 International Space Station (ISS) Agr...
3,Space Law,1998 ISS agreement,a researcher with strong analytical mind and e...,Can you provide specific examples of how the c...,"As of my last update, specific cases where the..."
4,Space Law,1998 ISS agreement,a professional in the domain seeking for techn...,What are the specific impacts of the 1998 ISS ...,The 1998 International Space Station (ISS) Agr...
...,...,...,...,...,...
2085,Telecommunication,Deep Space Communications,a college student who uses slang and abbreviat...,"Yo, so like, do antennas for deep space stuff ...","Yes, antennas for deep space communications of..."
2086,Telecommunication,Deep Space Communications,a college student who uses slang and abbreviat...,"Yo, so like, do these big antennas have to lik...","Yes, these big antennas are designed to move a..."
2087,Telecommunication,Deep Space Communications,a researcher with strong analytical mind and e...,Could you elucidate on the principles of forwa...,Forward Error Correction (FEC) is a method use...
2088,Telecommunication,Deep Space Communications,a researcher with strong analytical mind and e...,Could you provide examples of specific error-c...,"In deep space communication, specific error-co..."


In [3]:
qa_df["QandA"] = None
qa_df["QandA"] = qa_df.apply(lambda row: f"Q : {row['question']} A : {row['answer']}", axis=1)
qa_df["QandA"]

0       Q : Basics of Space Law? Role of international...
1       Q : Can you provide examples of specific natio...
2       Q : Could you provide a detailed analysis of t...
3       Q : Can you provide specific examples of how t...
4       Q : What are the specific impacts of the 1998 ...
                              ...                        
2085    Q : Yo, so like, do antennas for deep space st...
2086    Q : Yo, so like, do these big antennas have to...
2087    Q : Could you elucidate on the principles of f...
2088    Q : Could you provide examples of specific err...
2089    Q : Can you elaborate on how the Reed-Solomon ...
Name: QandA, Length: 2090, dtype: object

In [4]:
from transformers import AutoTokenizer
from sentence_transformers import SentenceTransformer
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
st_model = SentenceTransformer(MODEL_NAME)
print(tok)


c:\Users\skrrrt\miniconda3\envs\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


BertTokenizerFast(name_or_path='sentence-transformers/all-MiniLM-L6-v2', vocab_size=30522, model_max_length=512, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)


In [ ]:
TARGET_TOKENS = 500
OVERLAP_TOKENS = 80

step = TARGET_TOKENS - OVERLAP_TOKENS

chunk_rows = []

for i, row in qa_df.iterrows():
    ids = tok.encode(row["QandA"], truncation=True)
    for j, start in enumerate(range(0, len(ids), step)):
        piece = ids[start:start + TARGET_TOKENS]
        chunk = tok.decode(piece, skip_special_tokens=True)
        chunk_rows.append({
            "row_id": i,
            "part": j,
            "text": chunk,
            "topic": row.get("topic"),
            "subtopic": row.get("subtopic"),
            "persona": row.get("persona"),
        })


In [6]:
chunk_df = pd.DataFrame(chunk_rows)
chunk_df

,row_id,part,text,topic,subtopic,persona
0,0,0,q : basics of space law? role of international...,Space Law,Space Law Basics,"a busy person, needing quick and efficient ans..."
1,0,1,"objects, including debris. however, specific r...",Space Law,Space Law Basics,"a busy person, needing quick and efficient ans..."
2,1,0,q : can you provide examples of specific natio...,Space Law,Space Law Basics,"a busy person, needing quick and efficient ans..."
3,1,1,commercial endeavors. it establishes the gover...,Space Law,Space Law Basics,"a busy person, needing quick and efficient ans..."
4,2,0,q : could you provide a detailed analysis of t...,Space Law,1998 ISS agreement,a researcher with strong analytical mind and e...
...,...,...,...,...,...,...
3344,2087,0,q : could you elucidate on the principles of f...,Telecommunication,Deep Space Communications,a researcher with strong analytical mind and e...
3345,2088,0,q : could you provide examples of specific err...,Telecommunication,Deep Space Communications,a researcher with strong analytical mind and e...
3346,2088,1,data integrity in the challenging environment ...,Telecommunication,Deep Space Communications,a researcher with strong analytical mind and e...
3347,2089,0,q : can you elaborate on how the reed - solomo...,Telecommunication,Deep Space Communications,a researcher with strong analytical mind and e...


In [7]:
embs = st_model.encode(
    chunk_df["text"].tolist(),
    batch_size=64,
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=True,
)

print(f"Total chunk: {len(chunk_df)} | Embedding shape: {embs.shape}")

Batches: 100%|██████████| 53/53 [00:05<00:00,  9.93it/s]

Total chunk: 3349 | Embedding shape: (3349, 384)


In [8]:
import faiss

base_index = faiss.IndexHNSWFlat(384, 64, faiss.METRIC_INNER_PRODUCT)

base_index.hnsw.efConstruction = 200    # build quality
base_index.hnsw.efSearch = 128          # search quality

base_index.add(embs.astype("float32"))

In [9]:
import os
STORE_DIR = "faiss_store"
os.makedirs(STORE_DIR, exist_ok=True)

In [10]:
faiss.write_index(base_index, os.path.join(STORE_DIR, "index_hnsw_ip.faiss"))

chunk_df.to_parquet(os.path.join(STORE_DIR, "meta.parquet"), index=False)

In [11]:
# reload
index = faiss.read_index(os.path.join(STORE_DIR, "index_hnsw_ip.faiss"))
meta  = pd.read_parquet(os.path.join(STORE_DIR, "meta.parquet"))

q = "what is the most important space law?"

# [q] means only 1 vector, 1, 384 shape
q_embed = st_model.encode([q], normalize_embeddings=True, convert_to_numpy=True).astype("float32")

score, q_index = index.search(q_embed, 5)

print(score)
print(q_index)

[[0.5721501  0.53490746 0.53479576 0.5007783  0.49580416]]
[[2420   28   82 2391   39]]


In [12]:
for i in q_index[0]:
    print(meta.iloc[i]["text"])
    print("---")


of space operations.
---
notification and cooperation * * : states parties to the treaty shall keep each other informed about space activities and the discovery of any phenomena which could endanger human life or health in space. they are also encouraged to cooperate in the exploration and use of outer space. these principles form the foundation of international space law, promoting the peaceful use of outer space and ensuring that space activities are conducted for the benefit of all humanity.
---
q : what ' s the main stuff french space law is all about? a : french space law primarily governs the activities related to space exploration, utilization, and exploitation conducted by either public or private entities under french jurisdiction. it establishes a legal framework for authorization and licensing of space operations, outlines the liability for damage caused by space objects, and sets regulations for the registration of space objects. additionally, it emphasizes the importance o